# Phase 3: Target sweep, achievability, and label free correction

CPU only. Attach the same Phase 1 notebook output.

In [1]:
import glob, numpy as np, matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

p = [h for h in glob.glob("/kaggle/input/**/*.npz", recursive=True) if h.endswith("logits.npz")][0]
d = np.load(p); SEEDS=[0,1,2]
CORR = ["gaussian_noise","shot_noise","impulse_noise","defocus_blur","glass_blur",
        "motion_blur","zoom_blur","snow","frost","fog","brightness","contrast",
        "elastic_transform","pixelate","jpeg_compression"]
CELLS = [("test",0)] + [(c_,s) for s in range(1,6) for c_ in CORR]

def softmax(z,T=1.0):
    z=z.astype(np.float64)/T; z-=z.max(1,keepdims=True); e=np.exp(z); return e/e.sum(1,keepdims=True)
def nll(p,y): return -np.log(np.clip(p[np.arange(len(y)),y],1e-12,None)).mean()

def lg(key,seed): return d[f"{'test' if key=='test' else key}_{seed}"]
def ylab(key): return d["y_test"] if key=="test" else d["y_corr"]

# temperature fitted separately for single model and for ensemble
T1 = minimize_scalar(lambda t: nll(softmax(d["val_0"],t), d["y_val"]),
                     bounds=(.5,10), method="bounded").x
def ens_prob(mats,T): return np.mean([softmax(m,T) for m in mats],0)
TE = minimize_scalar(lambda t: nll(ens_prob([d[f"val_{s}"] for s in SEEDS],t), d["y_val"]),
                     bounds=(.5,10), method="bounded").x
print(f"T_single={T1:.3f}  T_ensemble={TE:.3f}")

def probs(key,sev,mode):
    k = "test" if key=="test" else f"{key}_{sev}"
    if mode=="single": return softmax(lg(k,0),T1)
    return ens_prob([lg(k,s) for s in SEEDS],TE)

P = {(k,s,m): (probs(k,s,m), ylab(k)) for k,s in CELLS for m in ("single","ens")}
PV = {"single": (softmax(d["val_0"],T1), d["y_val"]),
      "ens":    (ens_prob([d[f"val_{s}"] for s in SEEDS],TE), d["y_val"])}
print("ready")

T_single=1.468  T_ensemble=1.242
ready


## 1. Core helpers

`risk_cov` uses a small tolerance so a threshold that exactly meets the target is
not counted as a violation. That was a boundary artifact in Phase 2 (severity 0
reported 100% violated at risk 0.050).

In [2]:
TOL = 1e-9
def risk_cov(prob,y,thr):
    conf,pred = prob.max(1), prob.argmax(1); a = conf>=thr
    if a.sum()==0: return np.nan, 0.0
    return 1-(pred[a]==y[a]).mean(), a.mean()

def thr_for_risk(prob,y,target):
    """Lowest threshold whose realised risk <= target (max coverage). Needs labels."""
    conf = prob.max(1); corr = (prob.argmax(1)==y)
    o = np.argsort(-conf); cum = np.cumsum(corr[o])
    risk = 1 - cum/np.arange(1,len(o)+1)
    ok = np.where(risk <= target+TOL)[0]
    return 1.0 if len(ok)==0 else conf[o][ok[-1]]

def thr_labelfree(prob,target):
    """Same, but risk ESTIMATED from confidence (1-conf). No labels used."""
    conf = prob.max(1); o = np.argsort(-conf)
    est = np.cumsum(1-conf[o])/np.arange(1,len(o)+1)
    ok = np.where(est <= target+TOL)[0]
    return 1.0 if len(ok)==0 else conf[o][ok[-1]]

## 2. Target sweep

Phase 2 used a single 5% target, where the clean threshold was 0.46 and coverage
stayed above 88% everywhere. That makes the rejector nearly inert. Sweeping
stricter targets tests whether the failure is an artifact of a lax target.

In [3]:
TARGETS=[0.01,0.02,0.05,0.10]
rows=[]
for tg in TARGETS:
    for mode in ("single","ens"):
        pv,yv = PV[mode]; thr = thr_for_risk(pv,yv,tg)
        for sev in range(6):
            ks=[("test",0)] if sev==0 else [(c_,sev) for c_ in CORR]
            rc=[risk_cov(*P[(k,s,mode)],thr) for k,s in ks]
            r=np.nanmean([x[0] for x in rc]); cv=np.mean([x[1] for x in rc])
            rows.append((tg,mode,sev,thr,r,cv,np.mean([x[0]>tg+TOL for x in rc])))

print(f"{'tgt':>5} {'mode':>6} {'thr':>6} {'sev':>3} {'risk':>7} {'ratio':>6} {'cov':>6} {'viol':>5}")
for tg,mode,sev,thr,r,cv,v in rows:
    if mode=="single":
        print(f"{tg:>5.2f} {mode:>6} {thr:>6.3f} {sev:>3} {r:>7.3f} {r/tg:>6.1f}x {cv:>6.3f} {v*100:>4.0f}%")

  tgt   mode    thr sev    risk  ratio    cov  viol
 0.01 single  0.941   0   0.010    1.0x  0.837    0%
 0.01 single  0.941   1   0.038    3.8x  0.712  100%
 0.01 single  0.941   2   0.060    6.0x  0.633  100%
 0.01 single  0.941   3   0.086    8.6x  0.558  100%
 0.01 single  0.941   4   0.122   12.2x  0.476  100%
 0.01 single  0.941   5   0.191   19.1x  0.362  100%
 0.02 single  0.829   0   0.020    1.0x  0.908  100%
 0.02 single  0.829   1   0.064    3.2x  0.818  100%
 0.02 single  0.829   2   0.096    4.8x  0.758  100%
 0.02 single  0.829   3   0.137    6.8x  0.698  100%
 0.02 single  0.829   4   0.190    9.5x  0.627  100%
 0.02 single  0.829   5   0.288   14.4x  0.520  100%
 0.05 single  0.457   0   0.051    1.0x  0.991  100%
 0.05 single  0.457   1   0.122    2.4x  0.973  100%
 0.05 single  0.457   2   0.175    3.5x  0.960  100%
 0.05 single  0.457   3   0.234    4.7x  0.944  100%
 0.05 single  0.457   4   0.305    6.1x  0.923  100%
 0.05 single  0.457   5   0.420    8.4x  0.890 

**Read this as:** the ratio column is your headline number. Realised risk divided
by the risk the operator asked for. Anything above 1 is a broken guarantee.

## 3. Is the target even achievable?

Separate question from whether the threshold transfers. Here we cheat and use
labels on the shifted data to find the best possible threshold, then report the
coverage it costs. If achievable coverage collapses to near zero, the honest
conclusion is that no thresholding scheme can deliver that target under that
shift, regardless of how you pick the threshold.

In [4]:
TG=0.05
print(f"{'sev':>3} {'oracle cov':>11} {'clean-thr cov':>14} {'oracle risk':>12}")
pv,yv = PV["ens"]; thr_clean = thr_for_risk(pv,yv,TG)
for sev in range(6):
    ks=[("test",0)] if sev==0 else [(c_,sev) for c_ in CORR]
    oc,orisk,cc=[],[],[]
    for k,s in ks:
        pr,y = P[(k,s,"ens")]
        t = thr_for_risk(pr,y,TG); r,cv = risk_cov(pr,y,t)
        oc.append(cv); orisk.append(r); cc.append(risk_cov(pr,y,thr_clean)[1])
    print(f"{sev:>3} {np.mean(oc):>11.3f} {np.mean(cc):>14.3f} {np.nanmean(orisk):>12.3f}")

sev  oracle cov  clean-thr cov  oracle risk
  0       1.000          1.000        0.047
  1       0.833          0.999        0.050
  2       0.696          0.998        0.050
  3       0.584          0.997        0.050
  4       0.458          0.996        0.050
  5       0.264          0.995        0.049


## 4. Label free correction

Recompute the threshold on each shifted set using only confidence, no labels.
If model confidence were calibrated this would work. It is not, so the question
is how much of the gap it closes.

In [5]:
print(f"{'sev':>3} {'clean-thr risk':>15} {'labelfree risk':>15} {'lf cov':>8}")
for sev in range(6):
    ks=[("test",0)] if sev==0 else [(c_,sev) for c_ in CORR]
    a,b,cv=[],[],[]
    for k,s in ks:
        pr,y = P[(k,s,"ens")]
        a.append(risk_cov(pr,y,thr_clean)[0])
        t = thr_labelfree(pr,TG); r,c2 = risk_cov(pr,y,t); b.append(r); cv.append(c2)
    print(f"{sev:>3} {np.nanmean(a):>15.3f} {np.nanmean(b):>15.3f} {np.mean(cv):>8.3f}")

sev  clean-thr risk  labelfree risk   lf cov
  0           0.047           0.047    1.000
  1           0.118           0.071    0.890
  2           0.172           0.093    0.809
  3           0.233           0.122    0.728
  4           0.306           0.153    0.632
  5           0.423           0.219    0.491


## 5. Per class disparity

Phase 2 showed risk ranging from 0.055 (frog) to 0.945 (dog) on one corruption
while coverage stayed near 1.0 for every class. Confirm it is systematic.

In [6]:
names=["plane","car","bird","cat","deer","dog","frog","horse","ship","truck"]
R=np.zeros((10,len(CORR)))
for j,c_ in enumerate(CORR):
    pr,y = P[(c_,3,"ens")]; a = pr.max(1)>=thr_clean
    for i in range(10):
        m=(y==i)&a
        R[i,j]= 1-(pr[m].argmax(1)==y[m]).mean() if m.sum() else np.nan
order=np.argsort(np.nanmean(R,1))
print(f"{'class':>7} {'mean risk':>10} {'min':>7} {'max':>7}   (severity 3)")
for i in order:
    print(f"{names[i]:>7} {np.nanmean(R[i]):>10.3f} {np.nanmin(R[i]):>7.3f} {np.nanmax(R[i]):>7.3f}")
print(f"\nworst/best class ratio: {np.nanmean(R,1).max()/np.nanmean(R,1).min():.1f}x")

  class  mean risk     min     max   (severity 3)
   frog      0.133   0.032   0.391
   ship      0.175   0.024   0.651
   deer      0.190   0.036   0.537
  truck      0.207   0.019   0.638
  plane      0.212   0.028   0.848
    car      0.213   0.033   0.672
   bird      0.223   0.072   0.403
  horse      0.266   0.035   0.820
    cat      0.293   0.087   0.845
    dog      0.423   0.110   0.945

worst/best class ratio: 3.2x


## 6. Bootstrap intervals on the headline number

In [7]:
rng=np.random.default_rng(0)
def boot(pr,y,thr,B=200):
    out=[]
    for _ in range(B):
        i=rng.integers(0,len(y),len(y)); out.append(risk_cov(pr[i],y[i],thr)[0])
    return np.nanpercentile(out,[2.5,97.5])

for sev in [1,3,5]:
    rs=[]; los=[]; his=[]
    for c_ in CORR:
        pr,y=P[(c_,sev,"ens")]; rs.append(risk_cov(pr,y,thr_clean)[0])
        lo,hi=boot(pr,y,thr_clean); los.append(lo); his.append(hi)
    print(f"sev {sev}: risk {np.mean(rs):.3f}  95% CI [{np.mean(los):.3f}, {np.mean(his):.3f}]  target {TG}")

sev 1: risk 0.118  95% CI [0.112, 0.124]  target 0.05
sev 3: risk 0.233  95% CI [0.226, 0.241]  target 0.05
sev 5: risk 0.423  95% CI [0.413, 0.431]  target 0.05


## What to write down

1. Risk ratio at each target and severity (section 2). This is the paper's claim.
2. Whether the failure persists at strict targets or is specific to lax ones.
3. Achievable coverage (section 3), which separates "wrong threshold" from
   "impossible target".
4. How much the label free rule recovers (section 4). If it recovers little,
   that is the negative result worth publishing.
5. Per class ratio (section 5) as the secondary finding.